# UNESA Knowledge Graph - Interactive Analysis

Notebook ini untuk eksplorasi interaktif Knowledge Graph yang sudah dibuat.

**Prerequisites:**
- Knowledge Graph sudah dibuat dengan `build_knowledge_graph.py`
- Neo4j running di localhost:7687

## Setup & Connection

In [ ]:
from neo4j import GraphDatabase
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Connection parameters
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "rizkyyk123"
DATABASE = "datascience"

# Create driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(query, params=None):
    """Execute query and return results as DataFrame"""
    with driver.session(database=DATABASE) as session:
        result = session.run(query, params or {})
        data = [dict(record) for record in result]
        return pd.DataFrame(data) if data else pd.DataFrame()

# Test connection
try:
    df = run_query("RETURN 'Connected!' as message")
    print("✓", df['message'][0])
except Exception as e:
    print("✗ Connection failed:", str(e))

## 1. Database Overview

In [ ]:
# Get node counts
query = """
MATCH (n)
RETURN labels(n)[0] as node_type, count(n) as count
ORDER BY count DESC
"""

df_nodes = run_query(query)
print("\n=== NODE COUNTS ===")
display(df_nodes)

# Visualize
plt.figure(figsize=(12, 6))
plt.barh(df_nodes['node_type'], df_nodes['count'])
plt.xlabel('Count')
plt.title('Node Distribution in Knowledge Graph')
plt.tight_layout()
plt.show()

In [ ]:
# Get relationship counts
query = """
MATCH ()-[r]->()
RETURN type(r) as relationship_type, count(r) as count
ORDER BY count DESC
"""

df_rels = run_query(query)
print("\n=== RELATIONSHIP COUNTS ===")
display(df_rels)

# Visualize
plt.figure(figsize=(12, 8))
plt.barh(df_rels['relationship_type'], df_rels['count'])
plt.xlabel('Count')
plt.title('Relationship Distribution in Knowledge Graph')
plt.tight_layout()
plt.show()

## 2. Researcher Analytics

In [ ]:
# Top researchers by publications
query = """
MATCH (r:Researcher)-[:AUTHORS]->(a:Article)
RETURN r.name as researcher, 
       r.jabatan_akademik as position,
       r.nama_prodi as program,
       count(DISTINCT a) as publications,
       sum(CASE WHEN a.citations IS NOT NULL THEN a.citations ELSE 0 END) as total_citations
ORDER BY publications DESC
LIMIT 20
"""

df_top = run_query(query)
print("\n=== TOP 20 RESEARCHERS BY PUBLICATIONS ===")
display(df_top)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Publications
df_top_10 = df_top.head(10)
ax1.barh(df_top_10['researcher'], df_top_10['publications'])
ax1.set_xlabel('Publications')
ax1.set_title('Top 10 Researchers by Publication Count')
ax1.invert_yaxis()

# Citations
ax2.barh(df_top_10['researcher'], df_top_10['total_citations'], color='coral')
ax2.set_xlabel('Total Citations')
ax2.set_title('Top 10 Researchers by Total Citations')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Productivity by academic position
query = """
MATCH (r:Researcher)
OPTIONAL MATCH (r)-[:AUTHORS]->(a:Article)
WITH r.jabatan_akademik as position,
     count(DISTINCT r) as researcher_count,
     count(DISTINCT a) as total_articles
WHERE position IS NOT NULL
RETURN position,
       researcher_count,
       total_articles,
       toFloat(total_articles) / researcher_count as avg_articles
ORDER BY total_articles DESC
"""

df_position = run_query(query)
print("\n=== PRODUCTIVITY BY ACADEMIC POSITION ===")
display(df_position)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(df_position))
width = 0.35

ax.bar([i - width/2 for i in x], df_position['researcher_count'], width, label='Researchers', alpha=0.8)
ax.bar([i + width/2 for i in x], df_position['avg_articles'], width, label='Avg Articles', alpha=0.8)

ax.set_xlabel('Academic Position')
ax.set_ylabel('Count')
ax.set_title('Researchers and Average Publications by Position')
ax.set_xticks(x)
ax.set_xticklabels(df_position['position'], rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plt.show()

## 3. Collaboration Network

In [ ]:
# Strongest collaborations
query = """
MATCH (r1:Researcher)-[c:CO_AUTHOR_WITH]-(r2:Researcher)
WHERE c.collaboration_count > 3
RETURN r1.name as researcher1,
       r2.name as researcher2,
       c.collaboration_count as collaborations
ORDER BY collaborations DESC
LIMIT 20
"""

df_collab = run_query(query)
print("\n=== STRONGEST COLLABORATIONS ===")
display(df_collab)

# Collaboration network statistics
query_stats = """
MATCH (r:Researcher)
OPTIONAL MATCH (r)-[c:CO_AUTHOR_WITH]-(co:Researcher)
WITH r, count(DISTINCT co) as collaborator_count
RETURN 
    count(r) as total_researchers,
    sum(collaborator_count) / 2 as total_collaborations,
    avg(collaborator_count) as avg_collaborators,
    max(collaborator_count) as max_collaborators
"""

df_stats = run_query(query_stats)
print("\n=== COLLABORATION NETWORK STATISTICS ===")
display(df_stats)

## 4. Publication Trends

In [ ]:
# Publication trends over years
query = """
MATCH (a:Article)
WHERE a.year IS NOT NULL AND a.year >= 2015
RETURN a.year as year, 
       count(a) as publications,
       sum(CASE WHEN a.citations IS NOT NULL THEN a.citations ELSE 0 END) as citations
ORDER BY year
"""

df_trends = run_query(query)
print("\n=== PUBLICATION TRENDS ===")
display(df_trends)

# Visualize
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Publications over time
ax1.plot(df_trends['year'], df_trends['publications'], marker='o', linewidth=2, markersize=8)
ax1.set_xlabel('Year')
ax1.set_ylabel('Number of Publications')
ax1.set_title('Publication Trends Over Time')
ax1.grid(True, alpha=0.3)

# Citations over time
ax2.plot(df_trends['year'], df_trends['citations'], marker='s', linewidth=2, markersize=8, color='coral')
ax2.set_xlabel('Year')
ax2.set_ylabel('Total Citations')
ax2.set_title('Citation Trends Over Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Subject Area Analysis

In [ ]:
# Subject area distribution
query = """
MATCH (a:Article)-[:HAS_SUBJECT]->(sa:SubjectArea)
RETURN sa.name as subject_area,
       count(a) as article_count
ORDER BY article_count DESC
LIMIT 15
"""

df_subjects = run_query(query)
print("\n=== TOP 15 SUBJECT AREAS ===")
display(df_subjects)

# Visualize
plt.figure(figsize=(12, 8))
plt.barh(df_subjects['subject_area'], df_subjects['article_count'])
plt.xlabel('Number of Articles')
plt.title('Top 15 Research Subject Areas')
plt.tight_layout()
plt.show()

## 6. Program Comparison

In [ ]:
# Compare study programs
query = """
MATCH (s:StudyProgram)<-[:ASSIGNED_TO]-(r:Researcher)
OPTIONAL MATCH (r)-[:AUTHORS]->(a:Article)
OPTIONAL MATCH (r)-[:WORKS_ON]->(rp:ResearchProject)
WITH s,
     count(DISTINCT r) as researchers,
     count(DISTINCT a) as articles,
     count(DISTINCT rp) as projects
RETURN s.name as program,
       researchers,
       articles,
       projects,
       toFloat(articles) / researchers as articles_per_researcher
ORDER BY articles DESC
"""

df_programs = run_query(query)
print("\n=== PROGRAM COMPARISON ===")
display(df_programs)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Researchers
axes[0,0].barh(df_programs['program'], df_programs['researchers'])
axes[0,0].set_xlabel('Count')
axes[0,0].set_title('Researchers per Program')

# Articles
axes[0,1].barh(df_programs['program'], df_programs['articles'], color='coral')
axes[0,1].set_xlabel('Count')
axes[0,1].set_title('Articles per Program')

# Projects
axes[1,0].barh(df_programs['program'], df_programs['projects'], color='green')
axes[1,0].set_xlabel('Count')
axes[1,0].set_title('Research Projects per Program')

# Productivity
axes[1,1].barh(df_programs['program'], df_programs['articles_per_researcher'], color='purple')
axes[1,1].set_xlabel('Articles per Researcher')
axes[1,1].set_title('Research Productivity per Program')

plt.tight_layout()
plt.show()

## 7. Custom Query Playground

In [ ]:
# Write your own custom query here
custom_query = """
// Your Cypher query goes here
MATCH (r:Researcher)
RETURN r.name as name, r.jabatan_akademik as position
LIMIT 10
"""

df_custom = run_query(custom_query)
display(df_custom)

## 8. Export Results

In [ ]:
# Export all results to Excel
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'kg_analysis_{timestamp}.xlsx'

with pd.ExcelWriter(filename) as writer:
    df_nodes.to_excel(writer, sheet_name='Nodes', index=False)
    df_rels.to_excel(writer, sheet_name='Relationships', index=False)
    df_top.to_excel(writer, sheet_name='Top Researchers', index=False)
    df_position.to_excel(writer, sheet_name='By Position', index=False)
    df_collab.to_excel(writer, sheet_name='Collaborations', index=False)
    df_trends.to_excel(writer, sheet_name='Publication Trends', index=False)
    df_subjects.to_excel(writer, sheet_name='Subject Areas', index=False)
    df_programs.to_excel(writer, sheet_name='Programs', index=False)

print(f"✓ Results exported to {filename}")

## 9. Cleanup

In [ ]:
# Close connection
driver.close()
print("✓ Connection closed")